# 03_06 True Scale 3D Terrain Model

## 1.Introduction

This notebook constructs a three-dimensional terrain model of Waterloo Region using the clipped elevation model produced in the previous notebook.

The objective is to represent the terrain using consistent physical dimensions so that horizontal distances and vertical elevations can be compared directly.

The workflow includes:

Loading the clipped elevation model.
Examining spatial dimensions.
Converting all axes to common units.
Constructing coordinate grids.
Building a true-scale 3D terrain model.

Because Waterloo Region has relatively small elevation differences compared to its horizontal extent, the resulting model will appear nearly flat. This is expected and represents the true proportions of the landscape.

The following notebook will introduce vertical exaggeration to improve visual interpretation.

## 2. Imports

Import core geospatial and visualization libraries used in this notebook:

In [13]:
import rasterio
import numpy as np
import plotly.graph_objects as go

## 3. Load Clipped Elevation Model

The clipped DEM contains NoData values outside the valid elevation surface. These values are converted to NaN so that statistical calculations and visualizations ignore missing data.

In [14]:
clipped_path = "../../data/Elevation/clipped_dem.tif"

with rasterio.open(clipped_path) as src:

    z = src.read(1)

    bounds = src.bounds
    resolution = src.res
    nodata = src.nodata

z = np.where(z == nodata, np.nan, z)

## 4. Spatial Dimensions

Note that the dimensions for our x an y axes were previously listed as geographic coordinates using latitude and longitude. Now we want to convert those to km and get an estimate of the spatial size in familiar units of kilometers (km). Elevation would typically be in meters (m). We want to create a model with consistent units to make visualization and geomtric analysis easier and to make sure everythign is to scale. 

Note that when converting from degrees to km, the conversion depends on the latitude since the km per degree changes based on latitude. At the equator 1 degree of longitude is a greater distance than near the poles. So this is an approximation. You might have to change this depending on what area you are in. 

The clipped Waterloo Region elevation model bounding box covers approximately:

- Width: 55 km
- Height: 47 km
- Elevation range: approximately 191 m

Because the horizontal dimensions are much larger than the vertical relief, true-scale terrain models often appear nearly flat. This is a common characteristic of low-relief landscapes.

In [15]:
mean_lat = (bounds.top + bounds.bottom) / 2

km_per_deg_lat = 111.32
km_per_deg_lon = 111.32 * np.cos(np.radians(mean_lat))

width_km = (
    bounds.right - bounds.left
) * km_per_deg_lon

height_km = (
    bounds.top - bounds.bottom
) * km_per_deg_lat

z_min = np.nanmin(z)
z_max = np.nanmax(z)

z_range = z_max - z_min

print("Width (km):", round(width_km,2))
print("Height (km):", round(height_km,2))

print()

print("Minimum elevation (m):", round(z_min,1))
print("Maximum elevation (m):", round(z_max,1))
print("Elevation range (m):", round(z_range,1))

Width (km): 54.95
Height (km): 47.12

Minimum elevation (m): 245.0
Maximum elevation (m): 436.0
Elevation range (m): 191.0


## 5. Unit Conversion

The DEM stores elevation in meters, while the dataset bounds are expressed in geographic coordinates.

To construct a true-scale model, all three dimensions must use the same units.

For this notebook:

X axis = kilometers
Y axis = kilometers
Z axis = kilometers

Elevation values are therefore converted from meters to kilometers.

In [16]:
rows, cols = z.shape

x = np.linspace(0, width_km, cols)
y = np.linspace(0, height_km, rows)

z_km = z / 1000.0

## 6. Downsampling

Reduce sample rate to take every fourth row and column to make it easier to load, to rotate the graph and zoom. 

Downsampling changes the visual resolution but does not change the physical dimensions of the terrain model.

In [17]:
step = 4

z_small = z_km[::step, ::step]

x_small = x[::step]
y_small = y[::step]

## 7. Orientation Correction

Raster datasets often store rows beginning at the upper left corner of the image.

Plotly interprets matrix rows differently than rasterio, which can cause the terrain to appear mirrored.

Flipping the array vertically corrects the orientation while preserving all elevation values.
    
In my case the data was oriented the wrong way so I flipped it to mirror it along the x axis. 

In [18]:
z_small = np.flipud(z_small)

## 8. True Scale Terrain Model

In [19]:
fig = go.Figure(
    data=[
        go.Surface(
            x=x_small,
            y=y_small,
            z=z_small,
            colorscale="Earth"
        )
    ]
)

fig.update_layout(
    title="Waterloo Region True Scale Terrain Model",

    scene=dict(
        xaxis_title="Distance (km)",
        yaxis_title="Distance (km)",
        zaxis_title="Elevation (km)",

        aspectmode="manual",
        aspectratio=dict(
            x=width_km,
            y=height_km,
            z=z_range / 1000.0
        )
    )
)

fig.show()

## 9. Saving the Terrain Model

The true-scale terrain model created in this notebook is a Cartesian **visualization** rather than a traditional geospatial **dataset**.

The original DEM stores:

- X coordinates as longitude
- Y coordinates as latitude
- Z values as elevation in meters

For visualization purposes, this notebook converts the horizontal dimensions into kilometers and converts elevation into kilometers so that all three axes share the same units.

Although the resulting Plotly surface appears as a three-dimensional terrain model, these transformed coordinates exist only within the coordinate arrays created in this notebook and within the visualization itself.

The original raster file still stores:

- Geographic coordinates (latitude and longitude)
- A geographic coordinate reference system
- Elevation values referenced to the original DEM in meters

Saving the transformed visualization directly as a GeoTIFF would therefore produce a dataset whose coordinate information no longer matches its underlying spatial metadata.

The TIFF metadata still specifies:

- CRS = EPSG:4326
- Coordinates = longitude and latitude

Even though the Plotly X and Y coordinates have been converted to kilometers for visualization, the underlying raster still uses geographic coordinates.

Creating a true Cartesian terrain dataset would require constructing a new coordinate system and defining a new spatial transform for the raster. This process depends on the intended application and is beyond the scope of this notebook.

Because the coordinate conversion calculations are computationally inexpensive, it is generally simpler and safer to recompute the Cartesian terrain model whenever it is needed rather than saving an intermediate file.

The original clipped DEM remains the authoritative elevation dataset and should continue to be used for hydrological analysis and future terrain calculations.

The key takeaway is this important modeling principle:

- Raw data → save it.
- Cleaned data → save it.
- Reprojected data → save it.
- Clipped data → save it.
- Visualization transformations → usually recompute them.

That distinction will become increasingly important later when we begin constructing flow accumulation models, river networks, watershed analyses, and other derived products.